# ML Noise Generator Data Pipeline (Feature-Specific)
This notebook creates the dataset and analysis for physical sensor noise modeling across candidate viewpoints:
1. **Multi-Feature Masking**: Maps simulated CAD points and real scans to isolated geometric features (`feature0`, `feature1`, etc.).
2. **Pointwise Noise Extraction**: Computes 3D displacement vectors $(\Delta x, \Delta y, \Delta z)$, error magnitudes, sensor dropout rates, and Angle of Incidence (AoI) for every viewpoint.
3. **Correlation & 3D Visualization**: Analyzes the physical relationship between Angle of Incidence, reflective noise, and sensor dropouts.


In [ ]:
# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
import os
import re
import math
import copy
import glob
import time
import json
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.transform import Rotation as R

# Experiment and workpiece settings
EXPERIMENT = "test_10_realgrasp"
CAD_MODEL_DIR = "workpiece31"

# Directory paths
WORKPIECE_DIR = f"workpiece/{CAD_MODEL_DIR}"
WORKPIECE_CAD = f"workpiece/{CAD_MODEL_DIR}/workpiece.stl"
DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}"
if not os.path.exists(DATA_DIR):
    DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}/{CAD_MODEL_DIR}"

SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}"
if not os.path.exists(SIM_DIR):
    SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{CAD_MODEL_DIR}"

PROCESSED_DIR = f"processed_data/{EXPERIMENT}"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Registration and point cloud parameters
NUMBER_OF_POINTS = 40000
CROP_BOX = [85, 105, 70]  # [X, Y, Z] extent in mm

def get_rotation_matrix_z(deg):
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]])

def clean_and_crop_point_cloud(pcd, initial_pos, initial_angle, box_size, remove_plane=True, distance_threshold=3.0, plane_offset=0.5):
    if pcd.is_empty():
        return pcd
    pcd_clean = copy.deepcopy(pcd)
    if remove_plane:
        plane_model, inliers = pcd_clean.segment_plane(distance_threshold=distance_threshold, ransac_n=3, num_iterations=2000)
        [a, b, c, d] = plane_model
        pts = np.asarray(pcd_clean.points)
        distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
        above_plane_indices = np.where(distances > plane_offset)[0]
        pcd_clean = pcd_clean.select_by_index(above_plane_indices)
        
    rot_matrix = get_rotation_matrix_z(-initial_angle)
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos),
        R=rot_matrix,
        extent=np.array(box_size)
    )
    return pcd_clean.crop(obb)

# 1. Load Ground Truth Transformation (T_average without offset, valid row 4)
optitrack_path = os.path.join(DATA_DIR, "T_average.npy")
if not os.path.exists(optitrack_path):
    optitrack_path = os.path.join(DATA_DIR, "T_optitrack.npy")
if not os.path.exists(optitrack_path):
    eval_path = f"evaluation_result/{EXPERIMENT}/merge_full_transformation.npy"
    if os.path.exists(eval_path):
        optitrack_path = eval_path

T_gt = np.load(optitrack_path)
T_gt[3, :] = [0, 0, 0, 1]

if "T_optitrack" in optitrack_path:
    rad_z = math.radians(90.0)
    T_local_rot = np.eye(4)
    T_local_rot[:3, :3] = np.array([
        [math.cos(rad_z), -math.sin(rad_z), 0],
        [math.sin(rad_z),  math.cos(rad_z), 0],
        [0, 0, 1]
    ])
    T_gt = T_gt @ T_local_rot
    T_gt[3, :] = [0, 0, 0, 1]

# 2. Load YOLO Initial Guess
yolo_pose_path = os.path.join(DATA_DIR, "initial_obj_pose.npy")
if os.path.exists(yolo_pose_path):
    tf_obj = np.load(yolo_pose_path)
    YOLO_POS = tf_obj[:3].copy()
    YOLO_ANGLE = float(tf_obj[4])
    YOLO_POS[0] -= 0
    YOLO_POS[1] -= -10.0
else:
    YOLO_POS = [540.0, -70.0, 20.0]
    YOLO_ANGLE = 0.0

print(f"Loaded configuration for {EXPERIMENT} ({CAD_MODEL_DIR}):")
print(f"  - Ground Truth Pose: {optitrack_path}")
print(f"  - Ground Truth Translation: {np.round(T_gt[:3, 3], 2)}")
print(f"  - YOLO Position: {YOLO_POS} | Angle: {YOLO_ANGLE} deg")


In [ ]:
# ==========================================
# 2. MULTI-FEATURE MASKING & POINTWISE NOISE EXTRACTION
# ==========================================
# --- DEBUG / TESTING OPTIONS ---
DEBUG_MODE = False          # Set to True to test a subset of data for quick debugging
DEBUG_VIEWPOINTS = [226]   # Specific viewpoint IDs to test, e.g. [226] or [226, 137], or None for all
DEBUG_FEATURES = None      # Specific feature IDs to test, e.g. ["feature0"], or None for all
# -------------------------------

extracted_noise_data = []
MAX_NOISE_THRESHOLD = 15.0  # Max distance for matching point pairs (mm)
MASK_THRESHOLD = 20.0       # Feature isolation distance (mm)

# Discover feature files (supports feature*.stl and surface*.stl)
feature_files = sorted(glob.glob(os.path.join(WORKPIECE_DIR, "feature*.stl")) + glob.glob(os.path.join(WORKPIECE_DIR, "surface*.stl")))
feature_files = [f for f in feature_files if "background" not in os.path.basename(f)]

# Check CAD mesh center offset relative to global point cloud
global_pcd_path = os.path.join(SIM_DIR, "pcd_all.pcd")
cad_offset = np.zeros(3)
if os.path.exists(global_pcd_path) and os.path.exists(WORKPIECE_CAD):
    pcd_all = o3d.io.read_point_cloud(global_pcd_path)
    wp_mesh = o3d.io.read_triangle_mesh(WORKPIECE_CAD)
    if len(pcd_all.points) > 0:
        cad_offset = pcd_all.get_axis_aligned_bounding_box().get_center() - wp_mesh.get_axis_aligned_bounding_box().get_center()

# Find available viewpoint indices
candidate_files = glob.glob(os.path.join(SIM_DIR, "viewpoint_pose_*.npy"))
available_view_indices = []
for f in candidate_files:
    match = re.search(r'viewpoint_pose_(\d+)\.npy', f)
    if match:
        available_view_indices.append(int(match.group(1)))
available_view_indices = sorted(available_view_indices)

# Apply Debug Filtering
if DEBUG_MODE:
    if DEBUG_FEATURES is not None:
        feature_files = [f for f in feature_files if any(feat in os.path.basename(f) for feat in DEBUG_FEATURES)]
    if DEBUG_VIEWPOINTS is not None:
        available_view_indices = [v for v in available_view_indices if v in DEBUG_VIEWPOINTS]
    print(f"\n>>> [DEBUG MODE ACTIVE] Testing {len(available_view_indices)} viewpoints: {available_view_indices} on {len(feature_files)} features: {[os.path.basename(f) for f in feature_files]}\n")
else:
    print(f"Found {len(feature_files)} features: {[os.path.basename(f) for f in feature_files]}")
    print(f"Found {len(available_view_indices)} candidate viewpoints in {SIM_DIR}.")

if not feature_files:
    print(f"WARNING: No feature STL files found in {WORKPIECE_DIR}!")

start_time = time.time()

for feature_path in feature_files:
    feature_id = os.path.basename(feature_path).split('.')[0]
    print(f"\nProcessing Feature: {feature_id}...")
    
    mesh_feature = o3d.io.read_triangle_mesh(feature_path)
    if np.linalg.norm(cad_offset) > 1e-3:
        mesh_feature.translate(cad_offset)
    mesh_feature.compute_vertex_normals()
    pcd_feature_full = mesh_feature.sample_points_uniformly(number_of_points=10000)
    kdtree_feature = o3d.geometry.KDTreeFlann(pcd_feature_full)
    
    view_count = 0
    for view_idx in available_view_indices:
        sim_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{view_idx}.pcd")
        pose_path = os.path.join(SIM_DIR, f"viewpoint_pose_{view_idx}.npy")
        
        # Check real scan naming
        real_path = os.path.join(DATA_DIR, f"view{view_idx}.pcd")
        if not os.path.exists(real_path):
            real_path = os.path.join(DATA_DIR, f"view{view_idx:02d}.pcd")
        if not os.path.exists(real_path):
            real_path = os.path.join(PROCESSED_DIR, f"viewpoint_simulated_{view_idx}.pcd")
            
        if not os.path.exists(sim_path) or not os.path.exists(pose_path) or not os.path.exists(real_path):
            continue
            
        view_count += 1
        
        # 1. Load Real Scan and transform to Camera Frame
        raw_real = o3d.io.read_point_cloud(real_path)
        clean_real = clean_and_crop_point_cloud(raw_real, YOLO_POS, YOLO_ANGLE, CROP_BOX, remove_plane=True)
        
        T_cam_to_obj = np.load(pose_path)
        T_obj_to_cam = np.linalg.inv(T_cam_to_obj)
        T_target_to_object = np.linalg.inv(T_gt)
        
        clean_real.transform(T_target_to_object)
        clean_real.transform(T_obj_to_cam)
        
        # 2. Load Simulated Scan
        pcd_sim_down = o3d.io.read_point_cloud(sim_path)
        if not pcd_sim_down.has_normals():
            pcd_sim_down.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=10.0, max_nn=30))
            pcd_sim_down.orient_normals_towards_camera_location(camera_location=np.array([0., 0., 0.]))
            normals = np.asarray(pcd_sim_down.normals)
            pcd_sim_down.normals = o3d.utility.Vector3dVector(-normals)
            
        pcd_sim_down.transform(T_obj_to_cam)
        
        sim_points_cam = np.asarray(pcd_sim_down.points)
        sim_normals_cam = np.asarray(pcd_sim_down.normals)
        real_points_cam = np.asarray(clean_real.points)
        
        if len(sim_points_cam) == 0 or len(real_points_cam) == 0:
            continue
            
        # 3. Mask Simulated Points to Feature
        sim_points_obj = np.dot(sim_points_cam, T_cam_to_obj[:3, :3].T) + T_cam_to_obj[:3, 3]
        valid_sim_indices = []
        for i, pt_obj in enumerate(sim_points_obj):
            [k, idx, dist_sq] = kdtree_feature.search_knn_vector_3d(pt_obj, 1)
            if k > 0 and math.sqrt(dist_sq[0]) <= 2.0:
                valid_sim_indices.append(i)
                
        if len(valid_sim_indices) < 10:
            continue
            
        sim_points_final = sim_points_cam[valid_sim_indices]
        sim_normals_final = sim_normals_cam[valid_sim_indices]
        
        # 4. Mask Real Points to Feature
        real_points_obj = np.dot(real_points_cam, T_cam_to_obj[:3, :3].T) + T_cam_to_obj[:3, 3]
        valid_real_indices = []
        for i, pt_obj in enumerate(real_points_obj):
            [k, idx, dist_sq] = kdtree_feature.search_knn_vector_3d(pt_obj, 1)
            if k > 0 and math.sqrt(dist_sq[0]) <= MASK_THRESHOLD:
                valid_real_indices.append(i)
                
        if len(valid_real_indices) < 10:
            continue
            
        real_points_final = real_points_cam[valid_real_indices]
        
        # 5. Nearest-Neighbor Matching
        real_pcd_temp = o3d.geometry.PointCloud()
        real_pcd_temp.points = o3d.utility.Vector3dVector(real_points_final)
        kdtree_noise = o3d.geometry.KDTreeFlann(real_pcd_temp)
        
        edges = []
        for i in range(len(sim_points_final)):
            sim_pt = sim_points_final[i]
            [k, idx, dist_sq] = kdtree_noise.search_knn_vector_3d(sim_pt, 5)
            for j in range(k):
                dist = math.sqrt(dist_sq[j])
                if dist <= MAX_NOISE_THRESHOLD:
                    edges.append((dist, i, idx[j]))
                    
        edges.sort(key=lambda x: x[0])
        
        matched_sim = set()
        matched_real = set()
        
        for dist, sim_idx, real_idx in edges:
            if sim_idx not in matched_sim and real_idx not in matched_real:
                matched_sim.add(sim_idx)
                matched_real.add(real_idx)
                
                sim_pt = sim_points_final[sim_idx]
                sim_norm = sim_normals_final[sim_idx]
                real_pt = real_points_final[real_idx]
                
                # Angle of Incidence (AoI)
                camera_dir = -sim_pt
                camera_dir_norm = camera_dir / np.linalg.norm(camera_dir)
                normal_norm = sim_norm / np.linalg.norm(sim_norm)
                
                cos_theta = np.clip(np.dot(camera_dir_norm, normal_norm), -1.0, 1.0)
                aoi = np.degrees(np.arccos(cos_theta))
                if aoi > 90:
                    aoi = 180 - aoi

                extracted_noise_data.append({
                    'view_id': view_idx,
                    'feature_id': feature_id,
                    'sim_x': sim_pt[0],
                    'sim_y': sim_pt[1],
                    'sim_z': sim_pt[2],
                    'nx': sim_norm[0],
                    'ny': sim_norm[1],
                    'nz': sim_norm[2],
                    'aoi': aoi,
                    'dx': real_pt[0] - sim_pt[0],
                    'dy': real_pt[1] - sim_pt[1],
                    'dz': real_pt[2] - sim_pt[2],
                    'error_magnitude': dist,
                    'is_valid_scan': 1
                })
                
        # 6. Record Dropped / Occluded Points
        for sim_idx in range(len(sim_points_final)):
            if sim_idx not in matched_sim:
                sim_pt = sim_points_final[sim_idx]
                sim_norm = sim_normals_final[sim_idx]
                
                camera_dir = -sim_pt
                camera_dir_norm = camera_dir / np.linalg.norm(camera_dir)
                normal_norm = sim_norm / np.linalg.norm(sim_norm)
                
                cos_theta = np.clip(np.dot(camera_dir_norm, normal_norm), -1.0, 1.0)
                aoi = np.degrees(np.arccos(cos_theta))
                if aoi > 90:
                    aoi = 180 - aoi
                
                extracted_noise_data.append({
                    'view_id': view_idx,
                    'feature_id': feature_id,
                    'sim_x': sim_pt[0],
                    'sim_y': sim_pt[1],
                    'sim_z': sim_pt[2],
                    'nx': sim_norm[0],
                    'ny': sim_norm[1],
                    'nz': sim_norm[2],
                    'aoi': aoi,
                    'dx': 0.0,
                    'dy': 0.0,
                    'dz': 0.0,
                    'error_magnitude': 0.0,
                    'is_valid_scan': 0
                })

print(f"\nExtraction complete in {time.time() - start_time:.2f}s! Total points extracted: {len(extracted_noise_data)}")

df_noise = pd.DataFrame(extracted_noise_data)
csv_save_path = os.path.join(DATA_DIR, "extracted_noise_all_features.csv")
df_noise.to_csv(csv_save_path, index=False)
print(f"Saved dataset to: {csv_save_path}")


In [ ]:
# ==========================================
# 3. VISUALIZE SPECIFIC VIEWPOINT & NOISE VECTORS
# ==========================================
TARGET_VIEWPOINT = 226  # Change to any viewpoint index (e.g. 226, 137, 38)
TARGET_FEATURE = "all"  # Change to "feature0", "feature1", "feature2", "feature3", or "all"

print(f"Loading visualization for Viewpoint {TARGET_VIEWPOINT} | Feature: {TARGET_FEATURE}...")

csv_path = os.path.join(DATA_DIR, "extracted_noise_all_features.csv")
if not os.path.exists(csv_path):
    print(f"Dataset CSV not found at {csv_path}! Please run Section 2 first.")
else:
    df_noise = pd.read_csv(csv_path)
    df_filtered = df_noise[df_noise['view_id'] == TARGET_VIEWPOINT]
    if TARGET_FEATURE != "all":
        df_filtered = df_filtered[df_filtered['feature_id'] == TARGET_FEATURE]
        
    if len(df_filtered) == 0:
        print(f"No noise vectors found for viewpoint {TARGET_VIEWPOINT} and feature {TARGET_FEATURE}!")
    else:
        real_path = os.path.join(DATA_DIR, f"view{TARGET_VIEWPOINT}.pcd")
        if not os.path.exists(real_path):
            real_path = os.path.join(DATA_DIR, f"view{TARGET_VIEWPOINT:02d}.pcd")
        sim_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{TARGET_VIEWPOINT}.pcd")
        pose_path = os.path.join(SIM_DIR, f"viewpoint_pose_{TARGET_VIEWPOINT}.npy")
        
        if not os.path.exists(real_path) or not os.path.exists(sim_path):
            print(f"Point cloud files missing for view {TARGET_VIEWPOINT}!")
        else:
            pcd_sim = o3d.io.read_point_cloud(sim_path)
            raw_real = o3d.io.read_point_cloud(real_path)
            pcd_real = clean_and_crop_point_cloud(raw_real, YOLO_POS, YOLO_ANGLE, CROP_BOX, remove_plane=True)
            
            T_cam_to_obj = np.load(pose_path)
            T_obj_to_cam = np.linalg.inv(T_cam_to_obj)
            T_target_to_object = np.linalg.inv(T_gt)
            
            pcd_sim.transform(T_obj_to_cam)
            pcd_real.transform(T_target_to_object)
            pcd_real.transform(T_obj_to_cam)
            
            pcd_sim = pcd_sim.voxel_down_sample(voxel_size=1.0)
            pcd_real = pcd_real.voxel_down_sample(voxel_size=1.0)
            pcd_sim.paint_uniform_color([1.0, 0.0, 0.0])    # Red: Simulated CAD points
            pcd_real.paint_uniform_color([0.0, 0.65, 0.93]) # Blue: Real scanned points
            
            # Generate 3D Noise Vector Lines
            line_points, line_indices, line_colors = [], [], []
            cmap = plt.get_cmap("jet")
            
            for i, row in df_filtered[df_filtered['is_valid_scan'] == 1].iterrows():
                sim_pt = [row['sim_x'], row['sim_y'], row['sim_z']]
                real_pt = [row['sim_x'] + row['dx'], row['sim_y'] + row['dy'], row['sim_z'] + row['dz']]
                
                line_points.extend([sim_pt, real_pt])
                line_indices.append([len(line_points)-2, len(line_points)-1])
                
                norm_error = min(row['error_magnitude'] / 2.0, 1.0)
                line_colors.append(cmap(norm_error)[:3])
                
            line_set = o3d.geometry.LineSet()
            if line_points:
                line_set.points = o3d.utility.Vector3dVector(line_points)
                line_set.lines = o3d.utility.Vector2iVector(line_indices)
                line_set.colors = o3d.utility.Vector3dVector(line_colors)
                
            print(f"Rendering {len(line_indices)} noise vectors... (Red: CAD, Blue: Real Scan, Lines: Pointwise Noise Vector)")
            o3d.visualization.draw_geometries([pcd_sim, pcd_real, line_set], window_name=f"Noise - View {TARGET_VIEWPOINT} - Feature: {TARGET_FEATURE}")


In [ ]:
# ==========================================
# 4. CORRELATION ANALYSIS (AoI vs. Dropout vs. Noise)
# ==========================================
print("Loading dataset for Physical Noise & Dropout Correlation Analysis...")
csv_path = os.path.join(DATA_DIR, "extracted_noise_all_features.csv")
df_corr = pd.read_csv(csv_path)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(1, 2, figsize=(18, 6), dpi=120)

# 1. Pearson Correlation Heatmap
ax1 = axes[0]
features_to_correlate = ['sim_x', 'sim_y', 'sim_z', 'nx', 'ny', 'nz', 'aoi', 'dx', 'dy', 'dz', 'error_magnitude', 'is_valid_scan']
df_heatmap = df_corr[features_to_correlate]
corr_matrix = df_heatmap.corr(method='pearson')

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5, ax=ax1)
ax1.set_title("Feature Correlation Heatmap (Geometry vs. Noise)", fontsize=13, fontweight='bold')

# 2. Sensor Dropout Rate vs. Angle of Incidence (AoI)
ax2 = axes[1]
df_corr['aoi_bin'] = pd.cut(df_corr['aoi'], bins=range(0, 100, 10))
aoi_survival_rates = df_corr.groupby('aoi_bin', observed=False)['is_valid_scan'].mean() * 100.0
aoi_dropout_rates = 100.0 - aoi_survival_rates

sns.barplot(x=aoi_dropout_rates.index.astype(str), y=aoi_dropout_rates.values, color='salmon', ax=ax2)
ax2.set_title("Sensor Dropout Rate vs. Angle of Incidence (AoI)", fontsize=13, fontweight='bold')
ax2.set_xlabel("Angle of Incidence (Degrees)", fontsize=11)
ax2.set_ylabel("Dropout Rate (%)", fontsize=11)
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, "noise_aoi_correlation_analysis.png"), dpi=300)
plt.show()


In [ ]:
# ==========================================
# 5. VISUALIZE ANGLE OF INCIDENCE IN 3D
# ==========================================
TARGET_VIEWPOINT = 226  # Viewpoint index to inspect

print(f"Analyzing Angle of Incidence for Viewpoint {TARGET_VIEWPOINT}...")
csv_path = os.path.join(DATA_DIR, "extracted_noise_all_features.csv")
df_all = pd.read_csv(csv_path)
df_view = df_all[df_all['view_id'] == TARGET_VIEWPOINT].copy()

if len(df_view) == 0:
    print(f"No data found for viewpoint {TARGET_VIEWPOINT}")
else:
    # 1. Bar chart of AoI Distribution
    df_view['aoi_bin'] = pd.cut(df_view['aoi'], bins=range(0, 100, 10))
    aoi_counts = df_view['aoi_bin'].value_counts(observed=False).sort_index()
    
    plt.figure(figsize=(9, 4.5), dpi=120)
    sns.barplot(x=aoi_counts.index.astype(str), y=aoi_counts.values, color='skyblue')
    plt.title(f"Distribution of Angle of Incidence (Viewpoint {TARGET_VIEWPOINT})", fontsize=12, fontweight='bold')
    plt.xlabel("Angle of Incidence (Degrees)", fontsize=10)
    plt.ylabel("Number of Points", fontsize=10)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # 2. 3D Colormap of Points by AoI
    points = df_view[['sim_x', 'sim_y', 'sim_z']].values
    aoi_values = df_view['aoi'].values
    
    aoi_normalized = np.clip(aoi_values / 90.0, 0, 1)
    cmap = plt.get_cmap("jet")
    colors = cmap(aoi_normalized)[:, :3]
    
    pcd_aoi = o3d.geometry.PointCloud()
    pcd_aoi.points = o3d.utility.Vector3dVector(points)
    pcd_aoi.colors = o3d.utility.Vector3dVector(colors)
    
    print("\n--- 3D Color Map Legend ---")
    print("Blue   = 0 degrees (Face-on / Perpendicular ray)")
    print("Green  = ~45 degrees")
    print("Red    = 90 degrees (Grazing angle / High Noise)")
    print("Close the Open3D window to continue.")
    
    o3d.visualization.draw_geometries([pcd_aoi], window_name=f"AoI Colormap - View {TARGET_VIEWPOINT}")
